# YOLO Training

In [1]:
%pip install pytz pandas 

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
from logger import get_logger
logger = get_logger("YOLO-TRAIN")

2025-09-15 09:10 - INFO - Logger YOLO-TRAIN iniciado


## Environment

### Verificar CUDA en env

In [3]:
import torch

torch.cuda.is_available()

True

### Verificar librería ultralytics

In [4]:
# Issue por compatibilidad !!! 8.3.80
%pip install -qU ultralytics==8.3.80

Note: you may need to restart the kernel to use updated packages.


In [5]:
from ultralytics import __version__ as ultralytics_version

ultralytics_version

'8.3.80'

### Importar dependencias

Ingresamos la ruta del dataset

In [6]:
import os
import pandas as pd
from datetime import datetime
from ultralytics import YOLO

# DATASET_PATH = os.path.join(os.getcwd(), "data/desmodus_thermal.v1i.yolov11")
DATASET_PATH = os.path.join(os.getcwd(), "data/DESMUDIN.v1i.yolov11")

## YOLO series

Definimos los métodos de entrenamiento y exportación a formatos PT (PYTORCH) y TFLITE

Los hiperparámetros definidos son: 
device="cuda",
imgsz=640,
batch=16,
workers=64,
epochs=50,
pretrained=False,

In [ ]:
def train_yolo_model(model: YOLO, seed: int = 0):
    """Entrena el modelo yolo con el dataset de lissachatina"""
    res = model.train(
        data=os.path.join(DATASET_PATH, "data.yaml"),
        # optimizer="auto", # SGD, Adam, AdamW, NAdam, RAdam, RMSProp etc., or auto
        # lr0=0.01, #  (i.e. SGD=1E-2, Adam=1E-3)
        seed=seed,
        close_mosaic=True,
        device="cuda",
        imgsz=640,
        batch=16,
        # workers=64,
        workers=0,
        epochs=100,
        pretrained=False,
        patience=15,
    )

    return res


def export_yolo_model(model: YOLO) -> str:
    """Exporta el modelo yolo a tflite con Float16"""

    res_dir = model.export(
        format="tflite",
        half=True,
        # int8=True,
        imgsz=320,
        workers=0,
        # workers=64,
        device="cuda",
        data=os.path.join(DATASET_PATH, "data.yaml"),
    )

    return res_dir


def save_results_to_csv(trained_yolo_path: dict[tuple, str], name: str):
    """Guarda resultados de modelo, semilla y path en un csv"""
    df = pd.DataFrame(
        [
            {"yolo": yolo, "seed": seed, "path": path, "dataset": DATASET_PATH}
            for (yolo, seed), path in trained_yolo_path.items()
        ]
    )

    # Save to CSV
    name = name if name.endswith(".csv") else f"{name}.csv"
    df.to_csv(name, index=False)

### Train YOLO's (.pt)

In [8]:
trained_yolo_paths: dict[tuple, str] = {}

# Train for 2 different seeds
for seed in [3000]:
    for yolo in ["yolov8n", "yolov9t", "yolov10n", "yolo11n", "yolo12n"]:
        logger.info(f"Entrenando YOLO {yolo} con seed {seed}")

        yolo_model = YOLO(yolo)
        results = train_yolo_model(model=yolo_model, seed=seed)
        best_model_path = f"{str(results.save_dir)}/weights/best.pt"

        trained_yolo_paths[(yolo, seed)] = best_model_path
        logger.info("Guardado en %s", best_model_path)

TIMESTAMP = datetime.now().isoformat().replace(":", "-").replace(".", "-")
filename = f"{TIMESTAMP}_entrenamiento_yolo"
save_results_to_csv(trained_yolo_paths, filename)

logger.info("CSV file '%s' saved successfully.", filename)
trained_yolo_paths

2025-09-15 09:10 - INFO - Entrenando YOLO yolov8n con seed 3000


New https://pypi.org/project/ultralytics/8.3.199 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.80  Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24564MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=c:\Users\ialab\Documents\dev\desmodus-app\packages\notebooks\data/DESMUDIN.v1i.yolov11\data.yaml, epochs=100, time=None, patience=15, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=0, project=None, name=train11, exist_ok=False, pretrained=False, optimizer=auto, verbose=True, seed=3000, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=True, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnos

train: Scanning C:\Users\ialab\Documents\dev\desmodus-app\packages\notebooks\data\DESMUDIN.v1i.yolov11\train\labels... 1647 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1647/1647 [00:00<00:00, 1774.05it/s]


train: New cache created: C:\Users\ialab\Documents\dev\desmodus-app\packages\notebooks\data\DESMUDIN.v1i.yolov11\train\labels.cache


val: Scanning C:\Users\ialab\Documents\dev\desmodus-app\packages\notebooks\data\DESMUDIN.v1i.yolov11\valid\labels... 157 images, 0 backgrounds, 0 corrupt: 100%|██████████| 157/157 [00:00<00:00, 1891.46it/s]

...


100 epochs completed in 0.744 hours.
Optimizer stripped from runs\detect\train15\weights\last.pt, 5.5MB
Optimizer stripped from runs\detect\train15\weights\best.pt, 5.5MB

Validating runs\detect\train15\weights\best.pt...
Ultralytics 8.3.80  Python-3.11.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A5000, 24564MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:01<00:00,  3.59it/s]


                   all        157        191       0.99      0.998      0.993      0.847
Speed: 0.2ms preprocess, 1.7ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to runs\detect\train15


2025-09-15 12:22 - INFO - Guardado en runs\detect\train15/weights/best.pt
2025-09-15 12:22 - INFO - CSV file '2025-09-15T12-22-21-705462_entrenamiento_yolo' saved successfully.


{('yolov8n', 3000): 'runs\\detect\\train11/weights/best.pt',
 ('yolov9t', 3000): 'runs\\detect\\train12/weights/best.pt',
 ('yolov10n', 3000): 'runs\\detect\\train13/weights/best.pt',
 ('yolo11n', 3000): 'runs\\detect\\train14/weights/best.pt',
 ('yolo12n', 3000): 'runs\\detect\\train15/weights/best.pt'}

### Export YOLO's to .tflite

In [9]:
# exported_yolo_paths = {}

# for (model, seed), model_path in trained_yolo_paths.items():
#     logger.info("### Exportando modelo %s con seed %s desde %s...", model, seed, model_path)

#     yolo_model = YOLO(model_path)
#     res_dir = export_yolo_model(model=yolo_model)

#     exported_yolo_paths[(model, seed)] = res_dir
#     logger.info("Exportado en %s", res_dir)

# # Save to CSV
# TIMESTAMP = datetime.now().isoformat()
# filename = f"{TIMESTAMP}_exportado_yolo"
# save_results_to_csv(exported_yolo_paths, filename)

# logger.info("CSV file '%s' saved successfully.", filename)
# exported_yolo_paths